<a href="https://colab.research.google.com/github/Stdcoders/Graph-RAG/blob/main/GraphRAG_L5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/genaiconference/Agentic_KAG_Workshop_DHS_2026.git

Cloning into 'Agentic_KAG_Workshop_DHS_2026'...
remote: Enumerating objects: 403, done.
remote: Counting objects: 100% (179/179), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 403 (delta 125), reused 57 (delta 55), pack-reused 224 (from 2)
Receiving objects: 100% (403/403), 13.34 MiB | 19.43 MiB/s, done.
Resolving deltas: 100% (217/217), done.


In [ ]:
!pip install -r /content/Agentic_KAG_Workshop_DHS_2026/requirements.txt --quiet

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.7/263.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.0/358.0 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.8/85.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.4/669.4 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━

KeyboardInterrupt: 

In [ ]:
import os

os.chdir('/content/Agentic_KAG_Workshop_DHS_2026/')
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    print("error reading env details")
    pass

# --- Neo4j Sandbox ---
NEO4J_URI      = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE')

# --- OpenAI ---
os.environ.setdefault(
    'NVIDIA_API_KEY',
    os.getenv('NVIDIA_API_KEY')
)

print('NEO4J_URI :', NEO4J_URI)
print('NVIDIA key set:', bool(os.environ.get('NVIDIA_API_KEY')))

NEO4J_URI : neo4j+s://313964e6.databases.neo4j.io
NVIDIA key set: True


In [ ]:
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings.openai import BaseOpenAIEmbeddings

NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"

class NemotronEmbeddings(BaseOpenAIEmbeddings):
    """
    NVIDIA Nemotron embeddings via NIM's OpenAI-compatible /v1/embeddings endpoint.
    Nemotron-family embedding models require an `input_type` of "passage" (indexing)
    or "query" (retrieval) on every request, so we inject it automatically here.
    """
    def __init__(self, model="nvidia/nemotron-3-embed-1b", input_type="passage", **kwargs):
        self.input_type = input_type
        super().__init__(model=model, **kwargs)

    def _initialize_client(self, **kwargs):
        return self.openai.OpenAI(**kwargs)

    def embed_query(self, text, **kwargs):
        kwargs.setdefault("extra_body", {"input_type": self.input_type})
        return super().embed_query(text, **kwargs)


llm = OpenAILLM(
    model_name="nvidia/nemotron-3-super-120b-a12b",
    model_params={
        "response_format": {"type": "json_object"},
    },
    base_url=NVIDIA_BASE_URL,
    api_key=os.getenv("NVIDIA_API_KEY"),
)

embedder = NemotronEmbeddings(
    model="nvidia/nemotron-3-embed-1b",
    base_url=NVIDIA_BASE_URL,
    api_key=os.getenv("NVIDIA_API_KEY"),
    input_type="passage",
)

In [ ]:
from langfuse.langchain import CallbackHandler
from langfuse import get_client

os.environ["LANGFUSE_PUBLIC_KEY"] = os.getenv("LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_SECRET_KEY"] = os.getenv("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_HOST"] = (
    os.getenv("LANGFUSE_BASE_URL")
    or os.getenv("LANGFUSE_HOST")
    or "https://cloud.langfuse.com"
)

langfuse = get_client()

# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
    print(os.environ["LANGFUSE_HOST"])
else:
    print("Authentication failed. Please check your credentials and host.")

langfuse_handler = CallbackHandler()

Langfuse client is authenticated and ready!
https://cloud.langfuse.com


In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print('Connected to Neo4j ✔')

Connected to Neo4j ✔


In [ ]:
from neo4j_graphrag.indexes import create_vector_index, create_fulltext_index, drop_index_if_exists

In [ ]:
for _idx in ["graph_embedding_index", "graph_text_index"]:
    drop_index_if_exists(driver, _idx)

create_vector_index(
    driver=driver,
    name="graph_embedding_index",
    label="__Entity__",
    embedding_property="embedding",
    dimensions=1536,
    similarity_fn="cosine",
)

create_fulltext_index(
    driver=driver,
    name="graph_text_index",
    label="__Entity__",
    node_properties=["name", "description"],
)

In [ ]:
for _idx in ["movie_embedding_index", "movie_text_index"]:
    drop_index_if_exists(driver, _idx)

create_vector_index(
    driver=driver,
    name="movie_embedding_index",
    label="Movie",
    embedding_property="embedding",
    dimensions=2048,
    similarity_fn="cosine",
)

create_fulltext_index(
    driver=driver,
    name="movie_text_index",
    label="Movie",
    node_properties=["title", "overview", "tagline"],
)

In [ ]:
from helper_utils import pretty_retriever_results

In [ ]:
from neo4j_graphrag.retrievers import VectorRetriever

vector_retriever = VectorRetriever(
   driver,
   index_name="movie_embedding_index",
   embedder=embedder,
   return_properties=["title", "overview"],
)
vector_res = vector_retriever.get_search_results(query_text="A heartwarming animated film about family and growing up", top_k=5)

# Pretty-print the records as a tidy DataFrame
pretty_retriever_results(vector_res, title="Vector Retriever")

#### Vector Retriever  &nbsp;·&nbsp; _0 result(s)_

""


In [ ]:
EXPANSION_QUERY = """
WITH node AS movie, score
OPTIONAL MATCH (movie)-[:HAS_GENRE]->(g:Genre)
OPTIONAL MATCH (movie)<-[:CAST_IN]-(actor:Person)
OPTIONAL MATCH (movie)-[:TAGGED_WITH]->(k:Keyword)
OPTIONAL MATCH (movie)-[:PRODUCED_IN]->(c:Country)
RETURN
  movie.title AS title,
  coalesce(movie.overview, "") AS overview,
  score,
  collect(DISTINCT g.name)        AS genres,
  collect(DISTINCT actor.name)    AS cast,
  collect(DISTINCT k.name)        AS keywords,
  collect(DISTINCT c.name)        AS countries
ORDER BY score DESC"""

from neo4j_graphrag.retrievers import VectorCypherRetriever

vc_retriever = VectorCypherRetriever(
   driver,
   index_name="movie_embedding_index",
   embedder=embedder,
   retrieval_query = EXPANSION_QUERY,
)
vc_res = vc_retriever.get_search_results(query_text="A dark psychological thriller with an unreliable narrator?", top_k=3)

# Pretty-print the records as a tidy DataFrame
pretty_retriever_results(vc_res, title="Vector Cypher Retriever")

#### Vector Cypher Retriever  &nbsp;·&nbsp; _0 result(s)_

""


# Disadvantages of vector retrieval
1. Domain Specific Limitations.
2. Cannot be used for precise string matching.

In [ ]:
from neo4j_graphrag.retrievers import HybridRetriever

hybrid_retriever = HybridRetriever(
    driver=driver,
    vector_index_name="movie_embedding_index",
    fulltext_index_name="movie_text_index",
    embedder=embedder,
    return_properties=[
        "title",
        "overview",
        "vote_average",
        "release_date"
    ]
)

query_text = "An epic historical movie featuring a character named Bahubali"
retriever_result = hybrid_retriever.search(
    query_text=query_text,
    top_k=3
)

# Pretty-print the results as a tidy DataFrame
pretty_retriever_results(retriever_result, title="Hybrid Retriever Results")

#### Hybrid Retriever Results  &nbsp;·&nbsp; _0 result(s)_

""


In [ ]:
from neo4j_graphrag.retrievers import HybridCypherRetriever

hc_retriever = HybridCypherRetriever(
    driver=driver,
    vector_index_name="movie_embedding_index",
    fulltext_index_name="movie_text_index",
    retrieval_query=EXPANSION_QUERY,
    embedder=embedder,
)
query_text = "An epic historical movie featuring a character named Bahubali"
retriever_result = hc_retriever.search(query_text=query_text, top_k=5)

# Pretty-print the results as a tidy DataFrame
pretty_retriever_results(retriever_result, title="Hybrid Cypher Retriever — A dark superhero film with a chaotic joker villain")

#### Hybrid Cypher Retriever — A dark superhero film with a chaotic joker villain  &nbsp;·&nbsp; _0 result(s)_

""


In [ ]:
from neo4j_graphrag.retrievers import Text2CypherRetriever
from neo4j_graphrag.schema import get_schema

import prompts
from examples import examples

schema = get_schema(driver)

t2c_retriever = Text2CypherRetriever(
    driver=driver,
    llm=llm,
    neo4j_schema=schema,
    custom_prompt=prompts.custom_text2cypher_prompt,
    examples=examples,
)

In [ ]:
# Define a sample query in natural language
query_text1 = "List the top 10 highest-rated Hindi movies"

# Convert the natural language query to Cypher, execute it, and pretty-print
response = t2c_retriever.search(query_text=query_text1)

# The helper also renders the generated Cypher (from response.metadata['cypher'])
pretty_retriever_results(response, title="Text2Cypher Retriever — top 10 Hindi movies")

Text2CypherRetrievalError: Failed to get search result: Invalid input '"cypher"': expected 'ORDER BY', 'CALL', 'CREATE', 'LOAD CSV', 'DELETE', 'DETACH', 'FILTER', 'FINISH', 'FOR', 'FOREACH', 'INSERT', 'LET', 'LIMIT', 'MATCH', 'MERGE', 'NODETACH', 'OFFSET', 'OPTIONAL', 'REMOVE', 'RETURN', 'SET', 'SHOW', 'SKIP', 'TERMINATE', 'UNWIND', 'USE', 'WHEN', 'WITH' or '{' (line 2, column 3 (offset: 12))
"  "cypher": "MATCH (m:Movie)-[:SPOKEN_IN]->(l:Language) WHERE toLower(l.name) = 'hindi' RETURN m.title, m.vote_average ORDER BY m.vote_average DESC LIMIT 10""
   ^

In [ ]:
# Define a sample query in natural language
query_text2 = "Which 5 directors have directed the most movies"

# Convert the natural language query to Cypher, execute it, and pretty-print
response = t2c_retriever.search(query_text=query_text2)

# The helper also renders the generated Cypher (from response.metadata['cypher'])
pretty_retriever_results(response, title="Text2Cypher Retriever — Directors")


Text2CypherRetrievalError: Failed to get search result: Invalid input '"query"': expected 'ORDER BY', 'CALL', 'CREATE', 'LOAD CSV', 'DELETE', 'DETACH', 'FILTER', 'FINISH', 'FOR', 'FOREACH', 'INSERT', 'LET', 'LIMIT', 'MATCH', 'MERGE', 'NODETACH', 'OFFSET', 'OPTIONAL', 'REMOVE', 'RETURN', 'SET', 'SHOW', 'SKIP', 'TERMINATE', 'UNWIND', 'USE', 'WHEN', 'WITH' or '{' (line 2, column 3 (offset: 12))
"  "query": "MATCH (m:Movie)-[:DIRECTED_BY]->(p:Person) WITH p.name AS directorName, COUNT(m) AS movieCount ORDER BY movieCount DESC, directorName LIMIT 5 RETURN directorName, movieCount""
   ^

In [ ]:
from neo4j_graphrag.generation import GraphRAG
from IPython.display import display, Markdown
import json

def query_graph(retriever, llm, query_text, retriever_config):
  """Queries the graph using a GraphRAG pipeline.

  Args:
    retriever: The retriever to use.
    llm: The language model to use.
    query_text: The query text.
    retriever_config: The retriever configuration.

  Returns:
    The response from the GraphRAG pipeline.
  """
  # Initialize the RAG pipeline
  rag = GraphRAG(retriever=retriever, llm=llm)

  # Query the graph
  response = rag.search(query_text=query_text,
                        retriever_config=retriever_config,
                        return_context=True,
                        response_fallback="I can not answer this question because I have no relevant context.",
                        # config={"callbacks": [langfuse_handler]}
                        )
  return response.answer

In [ ]:
query_text = "List down movies similar to Interstellar?"
top_k_value = 5

# Get results from Vector Retriever using query_graph function
display(Markdown("### Vector Retriever Results (using GraphRAG)"))
response = query_graph(vector_retriever, llm, query_text, {"top_k": top_k_value})
display(Markdown(response))

# Get results from Vector Cypher Retriever using query_graph function
display(Markdown("### Vector Cypher Retriever Results (using GraphRAG)"))
response = query_graph(vc_retriever, llm, query_text, {"top_k": top_k_value})
display(Markdown(response))

# Get results from Text2Cypher Retriever using the query_graph function
display(Markdown("### Text2Cypher Retriever Results (using GraphRAG)"))
# response = query_graph(t2c_retriever, llm, query_text, {})
# display(Markdown(response))

# Get results from Hybrid Retriever using query_graph function
display(Markdown("### Hybrid Retriever Results (using GraphRAG)"))
response = query_graph(hybrid_retriever, llm, query_text, {"top_k": top_k_value})
display(Markdown(response))

# Get results from Hybrid Cypher Retriever using the query_graph function
display(Markdown("### Hybrid Cypher Retriever Results (using GraphRAG)"))
response = query_graph(hc_retriever, llm, query_text, {"top_k": top_k_value})
display(Markdown(response))

### Vector Retriever Results (using GraphRAG)

I can not answer this question because I have no relevant context.

### Vector Cypher Retriever Results (using GraphRAG)

I can not answer this question because I have no relevant context.

### Text2Cypher Retriever Results (using GraphRAG)

### Hybrid Retriever Results (using GraphRAG)

I can not answer this question because I have no relevant context.

### Hybrid Cypher Retriever Results (using GraphRAG)

I can not answer this question because I have no relevant context.